# Imports and Data Loading

In [145]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.pipeline import Pipeline


from feature_utils import generate_gabor_kernel, get_encoded_gabor_features, compute_average_rgb, get_mean_frequency
from feature_utils import get_sobel_features, get_gabor_features, generate_gabor_kernel, get_local_binary_pattern
from normalization_utils import crop_image
from data_utils import get_images, get_labels

In [62]:
with open('config.json') as config_file:
    config = json.load(config_file)

In [63]:
# Load Features
gb_features_df = pd.read_pickle(config['feature_locs']['task_a']['gabor'])
sobel_df = pd.read_pickle(config['feature_locs']['task_a']['sobel'])
rgb_df = pd.read_pickle(config['feature_locs']['task_a']['rgb'])
frequency_df = pd.read_pickle(config['feature_locs']['task_a']['frequency'])

In [132]:
features_df = (
    gb_features_df
    .merge(rgb_df.drop('disaster_label', axis=1), left_index=True, right_index=True)
    .merge(frequency_df[['mean frequency']], left_index=True, right_index=True)
    .merge(sobel_df.drop(['label', 'disaster_type'], axis=1), left_index=True, right_index=True)
)

# Model Evaluation

In [127]:
# Create folds for evaluation
k_fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=25)

In [156]:
# Logistic Regression
accuracies = []
for i, (train_idx, test_idx) in enumerate(k_fold.split(features_df.drop('disaster_label', axis=1), features_df['disaster_label'])):
    y_train, y_test = features_df['disaster_label'].iloc[train_idx], features_df['disaster_label'].iloc[test_idx]
    X_train, X_test = features_df.drop('disaster_label', axis=1).iloc[train_idx], features_df.drop('disaster_label', axis=1).iloc[test_idx]
    lr_model = Pipeline([
        ('scaler', RobustScaler()),
        ('pca', PCA(10)),
        ('estimator', LogisticRegression(random_state=25))
    ])
    lr_model.fit(X_train, y_train)

    pred = lr_model.predict(X_test)
    accuracy = accuracy_score(y_test, pred)
    accuracies.append(accuracy)
    print(f"Accuracy {i}: {accuracy:.3f}")
print(f"Total Accuracy: {np.mean(accuracies):.3f}")

Accuracy 0: 1.000
Accuracy 1: 0.963
Accuracy 2: 1.000
Accuracy 3: 1.000
Accuracy 4: 0.950
Total Accuracy: 0.982


In [160]:
# Random Forest
accuracies = []
for i, (train_idx, test_idx) in enumerate(k_fold.split(features_df.drop('disaster_label', axis=1), features_df['disaster_label'])):
    y_train, y_test = features_df['disaster_label'].iloc[train_idx], features_df['disaster_label'].iloc[test_idx]
    X_train, X_test = features_df.drop('disaster_label', axis=1).iloc[train_idx], features_df.drop('disaster_label', axis=1).iloc[test_idx]
    rf_model = RandomForestClassifier(n_estimators=1000)
    rf_model.fit(X_train, y_train)

    pred = rf_model.predict(X_test)
    accuracy = accuracy_score(y_test, pred)
    accuracies.append(accuracy)
    print(f"Accuracy {i}: {accuracy:.3f}")
print(f"Total Accuracy: {np.mean(accuracies):.3f}")

Accuracy 0: 0.988
Accuracy 1: 0.950
Accuracy 2: 0.975
Accuracy 3: 0.950
Accuracy 4: 0.963
Total Accuracy: 0.965


In [161]:
# AdaBoost
accuracies = []
for i, (train_idx, test_idx) in enumerate(k_fold.split(features_df.drop('disaster_label', axis=1), features_df['disaster_label'])):
    y_train, y_test = features_df['disaster_label'].iloc[train_idx], features_df['disaster_label'].iloc[test_idx]
    X_train, X_test = features_df.drop('disaster_label', axis=1).iloc[train_idx], features_df.drop('disaster_label', axis=1).iloc[test_idx]
    rf_model = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=1000
    )
    rf_model.fit(X_train, y_train)

    pred = rf_model.predict(X_test)
    accuracy = accuracy_score(y_test, pred)
    accuracies.append(accuracy)
    print(f"Accuracy {i}: {accuracy:.3f}")
print(f"Total Accuracy: {np.mean(accuracies):.3f}")

Accuracy 0: 1.000
Accuracy 1: 0.950
Accuracy 2: 1.000
Accuracy 3: 1.000
Accuracy 4: 0.963
Total Accuracy: 0.983


In [162]:
# XGBoost
accuracies = []
for i, (train_idx, test_idx) in enumerate(k_fold.split(features_df.drop('disaster_label', axis=1), features_df['disaster_label'])):
    y_train, y_test = features_df['disaster_label'].iloc[train_idx], features_df['disaster_label'].iloc[test_idx]
    X_train, X_test = features_df.drop('disaster_label', axis=1).iloc[train_idx], features_df.drop('disaster_label', axis=1).iloc[test_idx]
    rf_model = GradientBoostingClassifier(
        n_estimators=1000,
        learning_rate=1,
        max_depth=1,
    )
    rf_model.fit(X_train, y_train)

    pred = rf_model.predict(X_test)
    accuracy = accuracy_score(y_test, pred)
    accuracies.append(accuracy)
    print(f"Accuracy {i}: {accuracy:.3f}")
print(f"Total Accuracy: {np.mean(accuracies):.3f}")

Accuracy 0: 1.000
Accuracy 1: 0.963
Accuracy 2: 1.000
Accuracy 3: 1.000
Accuracy 4: 1.000
Total Accuracy: 0.993
